# RankAdaptation — α Sweep on L4 GPU
Tests L8 steering at multiple α values on Qwen2.5-7B-Instruct.

In [ ]:
# Install deps
!pip install -q transformers datasets bitsandbytes accelerate
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

# Clone repo
!git clone https://github.com/Filip-Miara/TrimTab.git
%cd TrimTab

In [ ]:
# Remove old .pt checkpoints from the repo (large files not tracked)
!rm -f best_*.pt

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Train standard TT on GSM8K test generations (data generated on-the-fly)
import torch, sys, os, re, math
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
sys.path.insert(0, '.')
from src.adapters.trajectory_transformer import TrajectoryTransformer

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print('Loading model (4-bit)...')
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True,
                                              quantization_config=quant, device_map='cuda')
model.eval()
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok.pad_token = tok.eos_token
print('Loaded')

# Clear GPU cache after model loading
torch.cuda.empty_cache()

In [ ]:
# Mount Google Drive for persistent trajectory cache
from google.colab import drive
drive.mount('/content/drive')
import glob
TRAJ_DIR = '/content/drive/MyDrive/TrimTab/trajs/'
os.makedirs(TRAJ_DIR, exist_ok=True)

device = 'cuda'; N_LAYERS = 28; D_MODEL = 3584

def extract_trajectories(input_ids, gen_len, hidden_states, n_layers):
    """Extract (hidden_seq, velocity) trajectories from batched hidden states.
    
    hidden_states: tuple of (B, L, D) per layer from one forward pass.
    Returns (h_seq, v_target) per item in the batch.
    """
    plen = input_ids.shape[1]
    n_positions = gen_len - 1  # number of velocity steps: (h[1]-h[0])...(h[gen-1]-h[gen-2])
    if n_positions < 1:
        return None, None
    seq_end = plen + gen_len
    # Shape: (n_layers, gen_len) → stack hidden states at positions plen..seq_end-1
    h_at_layers = torch.stack(
        [hs[0, plen:seq_end].float() for hs in hidden_states[:n_layers]], dim=0
    )  # (n_layers, gen_len, D)
    h_seq = h_at_layers[:, :-1]    # (n_layers, gen_len-1, D)
    h_next = h_at_layers[:, 1:]     # (n_layers, gen_len-1, D)
    v_target = h_next - h_seq       # (n_layers, gen_len-1, D)
    return h_seq.permute(1, 0, 2), v_target.permute(1, 0, 2)


if os.path.exists(f'{TRAJ_DIR}/done.flag'):
    print('Loading cached trajectories from Drive...')
    all_h, all_v = [], []
    for f in sorted(glob.glob(f'{TRAJ_DIR}/batch_*.pt')):
        d = torch.load(f)
        all_h.append(d['h']); all_v.append(d['v'])
    h = torch.cat(all_h); v = torch.cat(all_v)
    print(f'Loaded {len(h)} trajectories (float16, ~{h.numel()*2/1e9:.1f}GB)')
else:
    print('Collecting in batches of 8 for speed...')
    ds = load_dataset('openai/gsm8k', 'main', split='test')
    buf_h, buf_v, batch_idx = [], [], 0
    few_shot = ('Q: Janet has 5 ducks. She buys 3 more. How many?\nA: Step 1: 5. Step 2: buy 3. Step 3: 5+3=8. So answer is 8.\n\n'
                'Q: 12 muffins/hr, 8 hours?\nA: Step 1: 12/hr. Step 2: 8h. Step 3: 12x8=96. So answer is 96.\n\n')

    BATCH_SIZE = 8
    total = 300
    for i in range(0, total, BATCH_SIZE):
        batch_end = min(i + BATCH_SIZE, total)
        prompts = []
        for j in range(i, batch_end):
            prob = ds[j]
            prompts.append(f"{few_shot}Q: {prob['question']}\nA:")

        # Batch tokenize with padding
        inputs = tok(prompts, return_tensors='pt', padding=True, truncation=True).to(device)
        plen = inputs.input_ids.shape[1]

        # Batch generate
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False,
                             pad_token_id=tok.eos_token_id)

        # Single batched forward pass for hidden states
        with torch.no_grad():
            fwd = model(out, use_cache=False, output_hidden_states=True)
        hs = fwd.hidden_states  # tuple of (B, L', D) per layer, L' = out.shape[1]

        # Extract per-item trajectories
        for b in range(len(prompts)):
            # Find where this item's non-padding tokens end
            item_mask = out[b] != tok.pad_token_id
            gen_start = plen  # prompt length is uniform (padded to max)
            valid_positions = item_mask[gen_start:].sum().item()
            if valid_positions <= 1:
                continue

            # Get item-specific hidden states
            item_hs = [h[b:b+1, :gen_start+valid_positions] for h in hs]
            h_seq, v_seq = extract_trajectories(
                inputs.input_ids[b:b+1], valid_positions, item_hs, N_LAYERS)
            if h_seq is None:
                continue
            buf_h.append(h_seq.cpu()); buf_v.append(v_seq.cpu())

        if len(buf_h) >= 50 or batch_end >= total:
            fname = f'{TRAJ_DIR}/batch_{batch_idx:04d}.pt'
            torch.save({'h': torch.cat(buf_h).half(), 'v': torch.cat(buf_v).half()}, fname)
            buf_h, buf_v = [], []; batch_idx += 1
            print(f'  [{batch_end}/{total}] saved {os.path.basename(fname)}')

    open(f'{TRAJ_DIR}/done.flag', 'w').close()
    print('All trajectories saved to Drive. Reloading...')

    # Load for training
    all_h, all_v = [], []
    for f in sorted(glob.glob(f'{TRAJ_DIR}/batch_*.pt')):
        d = torch.load(f)
        all_h.append(d['h']); all_v.append(d['v'])
    h = torch.cat(all_h); v = torch.cat(all_v)
    print(f'Loaded {len(h)} trajectories (float16, ~{h.numel()*2/1e9:.1f}GB)')

# Shuffle + split
perm = torch.randperm(len(h))
n_val = len(h) // 10
tr_h, tr_v = h[perm[n_val:]].half(), v[perm[n_val:]].half()
va_h, va_v = h[perm[:n_val]].half(), v[perm[:n_val]].half()
del all_h, all_v, h, v

In [ ]:
# Train TT — data on CPU (float16), batches to GPU (float32)

v_mean = tr_v.float().mean(dim=(0, 1), keepdim=True)
v_std = tr_v.float().std(dim=(0, 1), keepdim=True) + 1e-8
tr_vn = (tr_v.float() - v_mean) / v_std
va_vn = (va_v.float() - v_mean) / v_std

tt = TrajectoryTransformer(d_model=768, n_layers=6, n_heads=8, d_ff=3072,
                            n_positions=N_LAYERS, d_input=D_MODEL).to(device)
opt = torch.optim.AdamW(tt.parameters(), lr=5e-4)
bs = 256
best_r2 = -float('inf')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB')

for ep in range(20):
    tt.train()
    perm = torch.randperm(len(tr_h))
    for i in range(0, len(tr_h), bs):
        idx = perm[i:i+bs]
        vp = tt(tr_h[idx].float().to(device))
        loss = torch.nn.functional.mse_loss(vp, tr_vn[idx].float().to(device))
        opt.zero_grad(); loss.backward(); opt.step()

    tt.eval()
    with torch.no_grad():
        vp = tt(va_h.float().to(device))
        mse = (vp - va_vn.float().to(device)).pow(2).mean().item()
        r2 = 1 - mse / max(va_vn.var().item(), 1e-8)
    if r2 > best_r2:
        best_r2 = r2
        torch.save(tt.state_dict(), 'best_gen_tt_7b.pt')
    print(f'ep={ep+1:2d} val_r2={r2:.4f}', flush=True)
print(f'Best R²={best_r2:.4f} — TT saved')

In [ ]:
!python run_alpha_sweep.py --n-test 100 --layer 8 --alphas -0.5 -0.2 -0.1 0.0 0.05 0.1 0.2 0.5 1.0 2>&1

In [ ]:
# Show results
import json
results = json.load(open('alpha_sweep_L8.json'))
baseline = results['baseline']
print(f"{'Alpha':>6} {'Acc':>8} {'Delta':>8}")
print(f"{'-----':>6} {'---':>8} {'-----':>8}")
for a_str, acc in sorted(results['alphas'].items()):
    a = float(a_str)
    d = acc - baseline if a != 0.0 else 0
    print(f"{a:>6.2f} {100*acc:>7.1f}% {100*d:>+7.1f}pp")

# Download results
from google.colab import files
files.download('alpha_sweep_L8.json')